<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L07-post-market-monitoring/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/ai-act-conformity/lessons/P01-L07-post-market-monitoring/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/ai-act-conformity/lessons/P01-L07-post-market-monitoring/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/ai-act-conformity/lessons/P01-L07-post-market-monitoring/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P01-L07 · Post-market monitoring and serious incidents

**You will build:** a post-market monitoring feed on top of the Article 12 log you built in
P01-L01 — control limits computed from a declared baseline, a windowed monitor that pages on
a breach, a deployer-ingest path that reconciles two clocks, and an Article 73 incident log
whose reporting clock starts at **awareness**, not at occurrence.

**Time:** ~80 minutes · **Runs on:** a laptop CPU, no download, no network
· **Prerequisites:** `T10-L01-ai-act-conformity-pack`, `P01-L01-article-12-logging`

Article 72 asks a provider to *actively and systematically* collect, document and analyse
performance data, including data a deployer sends back. Article 73 asks for serious
incidents to be reported on a clock. Between those two sentences sits the thing nobody
writes down: a monitor that pages on every wobble is muted within a month, and a monitor
that never pages is indistinguishable from no monitor at all. This lesson makes you compute
the cost of both, on the same feed, and pick a setting with a number behind it.

By the end you will be able to:

1. Implement p-chart control limits from a declared baseline window, and measure what
   happens to them when they are refitted over the period that contains the breach.
2. Implement a tumbling-window monitor that cuts its windows on the log's sequence number,
   drops the incomplete tail, and confirms a breach before it pages.
3. Measure both costs of a mis-tuned monitor — the pages raised for nothing, and the
   degraded windows nobody saw — and find the limit width that minimises their sum.
4. Implement a deployer-ingest path that estimates the clock offset by median rather than
   mean, reports its residual spread, and merges without counting the overlap twice.
5. Implement the Article 73 classifier and its clock, and explain why a fundamental-rights
   infringement does **not** get the two-day deadline.

> **This is engineering, not legal advice.** The article numbers, the quoted wording and the
> dates are sourced in `claims.yaml` with their URLs and access dates. Everything else —
> the 50-decision window, the run-of-two confirmation rule, the three-false-alerts mute
> rule, the two cost constants — are **this lesson's modelling choices**, argued for where
> they appear. They are not statements about what any authority would accept. For a real
> system, read the Official Journal text and take professional advice.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import hashlib
import io
import math
import statistics
import sys
import traceback
from datetime import date, datetime, timedelta, timezone
from typing import Callable

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__)

# The moment this monitoring report is cut. Fixed, so every clock below is reproducible: an
# incident log whose deadlines move with the calendar is not an artefact anyone can review.
AS_OF = date(2026, 9, 16)
AS_OF_DT = datetime(2026, 9, 16, 12, 0, tzinfo=timezone.utc)
SYSTEM_ID = "loan-copilot"


def iso(moment: datetime) -> str:
    """An ISO-8601 UTC timestamp with a trailing Z. Given to you; not graded."""
    return moment.isoformat().replace("+00:00", "Z")


def parse_ts(value: str) -> datetime:
    """Parse an ISO-8601 timestamp, accepting a trailing 'Z' or '+00:00'. Same helper as
    P01-L01; the two lessons read the same log format on purpose."""
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


print("as of", iso(AS_OF_DT), "· system", SYSTEM_ID)

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("control_limits",),
    "exercise 2": ("windowed_monitor",),
    "exercise 3": ("monitor_costs",),
    "exercise 4": ("estimate_clock_offset",),
    "exercise 5": ("merge_deployer_feed",),
    "exercise 6": ("classify_incident",),
    "exercise 7": ("incident_log",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (<its function>)"; several -> "exercises 3, 6 and 7"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What Articles 72 and 73 ask for

Article 72(1) asks for a post-market monitoring **system**, established and documented.
72(2) says what it has to do — actively and systematically collect, document and analyse
relevant data, *which may be provided by deployers or collected through other sources*.
That clause is the reason half this lesson is an ingest path: the data you need is not all
yours, and the party that holds it keeps its own clock.

Article 73 is the other half. It does not ask you to notice an incident; it assumes you
already have, and starts a clock. Each quoted sentence below is in `claims.yaml` against
its source URL and access date.

In [ ]:
DUTIES = {
    "art72_1_system": ("Article 72(1)",
                       "establish and document a post-market monitoring system"),
    "art72_2_collect": ("Article 72(2)",
                        "actively and systematically collect, document and analyse data, "
                        "which may be provided by deployers"),
    "art72_3_plan": ("Article 72(3)",
                     "the system rests on a post-market monitoring PLAN, part of Annex IV"),
    "art26_5_deployer": ("Article 26(5)",
                         "deployers monitor, inform the provider under Article 72, and on a "
                         "serious incident inform the provider first"),
    "art73_1_report": ("Article 73(1)",
                       "report any serious incident to the market surveillance authorities"),
    "art73_2_clock": ("Article 73(2)",
                      "immediately on a causal link, and in any event <= 15 days after "
                      "becoming AWARE"),
    "art73_3_short": ("Article 73(3)",
                      "widespread infringement, or Article 3(49)(b): <= 2 days after "
                      "becoming aware"),
    "art73_4_death": ("Article 73(4)",
                      "death of a person: <= 10 days after becoming aware"),
}
print(f"{'duty':22s} {'article':18s} what it requires")
for _key, (_article, _what) in DUTIES.items():
    print(f"{_key:22s} {_article:18s} {_what}")
print(f"\n{len(DUTIES)} duties. Note that {sum(1 for k in DUTIES if 'art73' in k)} of them are "
      "Article 73, and that they set three DIFFERENT deadlines.")

### Three deadlines, one starting gun

The deadlines are 15, 2 and 10 days, and the short one has a specific trigger: a
**widespread infringement**, or a serious incident as defined in **Article 3(49)(b)** — a
serious and irreversible disruption of the management or operation of critical
infrastructure. A fundamental-rights infringement is Article 3(49)(**c**) and gets the
ordinary 15 days. Guessing that the rights case must be the urgent one is the single most
common mistake in this area, and it is wrong in the expensive direction: you will also
guess that the critical-infrastructure case has a fortnight.

All three run from **awareness**. Not from the occurrence, not from the day you finished
the root-cause analysis.

In [ ]:
ARTICLE_3_49 = {
    "a": "the death of a person, or serious harm to a person's health",
    "b": "a serious and irreversible disruption of the management or operation of "
         "critical infrastructure",
    "c": "the infringement of obligations under Union law intended to protect "
         "fundamental rights",
    "d": "serious harm to property or the environment",
}
DEADLINE_DAYS = {"73(2)": 15, "73(3)": 2, "73(4)": 10}

for _point, _text in ARTICLE_3_49.items():
    print(f"3(49)({_point})  {_text}")
print()
for _para, _days in DEADLINE_DAYS.items():
    print(f"Article {_para}: {_days:2d} days from AWARENESS")
print(f"\nThe spread between the shortest and the longest is a factor of "
      f"{max(DEADLINE_DAYS.values()) / min(DEADLINE_DAYS.values()):.1f}. Getting the branch "
      "wrong is not a rounding error.")

### What the Digital Omnibus moved here, and what it did not

Regulation (EU) 2026/1744 amended Article 72(3): the **implementing act** that was to lay
down a post-market monitoring plan template by 2 February 2026 became Commission
**guidance, including a template**, by 2 September 2027. Article 73 it did not touch at all.

There is a wrinkle worth noticing. Article 113 as amended pushes Chapter III Sections 1-3 —
the requirements your monitor is measuring compliance *with* — out to 2 December 2027 and
2 August 2028. Chapter IX, which holds Articles 72 and 73, is not in that list of
exceptions, so on the face of the article it runs from the general date. That is a reading
of a list, not a ruling, and the lesson labels it as one.

In [ ]:
OMNIBUS_NOTES = {
    "art72_3": "implementing act + template by 2026-02-02 REPLACED with Commission guidance "
               "including a template by 2027-09-02",
    "art73": "not amended; the 15/2/10-day clocks are as adopted in 2024",
    "2027-12-02": "Chapter III Sections 1-3 apply, Annex III stand-alone high-risk",
    "2028-08-02": "Chapter III Sections 1-3 apply, Annex I product-embedded high-risk",
}
for _key, _what in OMNIBUS_NOTES.items():
    _state = "date" if _key[0].isdigit() else "text"
    print(f"{_key:12s} [{_state}] {_what}")
print("\nSo: the duty to watch, and the duty to report what you see, are not the parts of "
      "this Regulation that moved.")

## 2. The feed — synthetic, generated here, defective on purpose

Nothing is loaded from disk and nothing is downloaded. Both feeds below are **synthetic**,
built in this cell from one seed and an explicit per-window error count, so two runs of this
notebook agree to the last digit and every number you see is one your machine computed.

The provider feed is one row per scored decision, in the shape P01-L01's log emits: a
sequence number, a decision id, a timestamp, and whether the decision turned out to be
wrong. The deployer feed is what a partner sends back under Article 26(5) — some of the same
decisions, plus decisions you have no record of, on a clock that is not yours.

There are five defects, and the cell below builds every one of them in the open. That is
deliberate: you are meant to MEASURE each one, not read its size off a constant, which is
the only one of the two habits that survives contact with a feed nobody planted.

In [ ]:
WINDOW_SIZE = 50                 # decisions per monitoring window — this lesson's choice
BASELINE_WINDOWS = 16            # the declared baseline: windows 0..15, fixed before the fact
MIN_RUN = 2                      # consecutive out-of-limit windows before the monitor pages
DEFAULT_K = 3.0                  # the textbook control-limit width, in sigmas
K_GRID = (1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0)
SWITCH_OFF_AFTER = 3             # false pages after which a real team mutes the monitor
COST_PER_FALSE_ALERT = 400       # one out-of-hours investigation that found nothing
COST_PER_MISSED_WINDOW = 2500    # 50 decisions taken at a degraded error rate, unremediated
OUTLIER_TOLERANCE_SECONDS = 300  # a clock pair further than this from the median is suspect
DECISION_INTERVAL_HOURS = 3

# The per-window error counts. Explicit rather than drawn, so the whole lesson is
# reproducible and every figure below is a property of the fixture rather than of a seed.
WINDOW_ERRORS = (
    # windows 0-15: the declared baseline. 48 errors in 800 decisions, exactly 0.06.
    3, 2, 4, 3, 3, 2, 4, 3, 2, 4, 3, 3, 2, 4, 3, 3,
    # windows 16-25: in service. Three wobbles of rising size, none of them a fault.
    5, 5, 3, 6, 6, 2, 7, 7, 3, 2,
    # windows 26-30: the degradation this monitor exists to find.
    7, 7, 11, 12, 10,
    # windows 31-33: recovered.
    4, 4, 3,
)
TAIL_ROWS = 17                   # decisions after the last complete window
TAIL_ERRORS = 6
DEGRADED_WINDOWS = (26, 27, 28, 29, 30)  # ground truth — a fixture has one; a live monitor does not
_SEED = 20270302
print(f"{len(WINDOW_ERRORS)} full windows of {WINDOW_SIZE} + a tail of {TAIL_ROWS} = "
      f"{len(WINDOW_ERRORS) * WINDOW_SIZE + TAIL_ROWS} decisions")

In [ ]:
def _build_feeds() -> tuple:
    """Build the provider log and the deployer feed. Deterministic: one seed, fixed counts.

    Everything that makes the fixture defective — the buffered batch, the clock offset, the
    re-export, the decisions the provider never saw — is local to this function on purpose.
    You are meant to measure them, not read them off a constant.
    """
    rng = np.random.default_rng(_SEED)
    n_full = len(WINDOW_ERRORS) * WINDOW_SIZE
    n_total = n_full + TAIL_ROWS
    wrong = np.zeros(n_total, dtype=bool)
    for w, errors in enumerate(WINDOW_ERRORS):
        wrong[w * WINDOW_SIZE + rng.choice(WINDOW_SIZE, size=errors, replace=False)] = True
    wrong[n_full + rng.choice(TAIL_ROWS, size=TAIL_ERRORS, replace=False)] = True

    # A broken exporter buffered thirty decisions and flushed them ten days late, so their
    # timestamps bunch together long after the decisions were taken. The sequence numbers are
    # still right; only the clock is wrong. This is P01-L01's clock-skew entry, at scale.
    flush_from, flush_to, flush_hours = 1150, 1180, 241
    provider = []
    for i in range(n_total):
        stamp = AS_OF_DT - timedelta(hours=DECISION_INTERVAL_HOURS * (n_total - 1 - i))
        if flush_from <= i < flush_to:
            first = AS_OF_DT - timedelta(
                hours=DECISION_INTERVAL_HOURS * (n_total - 1 - flush_from))
            stamp = first + timedelta(hours=flush_hours, minutes=i - flush_from)
        provider.append({"seq": i, "decision_id": f"D-{i:05d}", "ts": iso(stamp),
                         "wrong": bool(wrong[i]), "source": "provider"})

    # The deployer's feed. Its clock reads local time and is labelled UTC, so every row is
    # offset; four rows were re-exported after somebody corrected a date by hand.
    offset, over_from, over_to = 7620, 1600, 1700
    n_only, n_only_wrong = 60, 40
    deployer = []
    for j, i in enumerate(range(over_from, over_to)):
        jitter = int(rng.integers(-4, 5))
        extra = 86400 if j % 25 == 7 else 0
        stamp = parse_ts(provider[i]["ts"]) + timedelta(seconds=offset + jitter + extra)
        deployer.append({"decision_id": provider[i]["decision_id"], "ts": iso(stamp),
                         "wrong": provider[i]["wrong"], "source": "deployer"})
    only_wrong = {int(x) for x in rng.choice(n_only, size=n_only_wrong, replace=False)}
    start = parse_ts(provider[over_from]["ts"])
    span = (parse_ts(provider[n_total - 1]["ts"]) - start).total_seconds()
    for j in range(n_only):
        stamp = start + timedelta(seconds=span * (j + 0.5) / n_only + offset)
        deployer.append({"decision_id": f"P-{j:04d}", "ts": iso(stamp),
                         "wrong": j in only_wrong, "source": "deployer"})
    return provider, deployer


PROVIDER_FEED, DEPLOYER_FEED = _build_feeds()
BASELINE_ERRORS = sum(WINDOW_ERRORS[:BASELINE_WINDOWS])
BASELINE_N = BASELINE_WINDOWS * WINDOW_SIZE
print(f"provider feed {len(PROVIDER_FEED)} rows · deployer feed {len(DEPLOYER_FEED)} rows")
print(f"declared baseline: windows 0..{BASELINE_WINDOWS - 1}, {BASELINE_ERRORS} errors in "
      f"{BASELINE_N} decisions")
print(f"first row: {PROVIDER_FEED[0]}")
print(f"a deployer row: {DEPLOYER_FEED[0]}")

## 3. Exercise 1 — control limits, and where the baseline comes from

A monitoring plan that says "alert if the error rate gets worse" is not a plan. It has to
say: worse than what, measured over how many decisions, and how far outside before anyone
is woken up. That is a **p-chart**: a centre line at the baseline rate `p`, and limits at
`k` standard errors either side, where the standard error of a proportion measured over `n`
decisions is `sqrt(p * (1 - p) / n)`.

Two things about that formula decide whether the chart is honest. The `n` is the **window
size**, not the baseline size — the limits have to describe the spread of the thing you
will plot. And the baseline period is declared **before** you look, because limits refitted
over the period that contains the breach are limits wide enough to contain it. That is the
defect the capstone module plants, and you are about to reproduce it on purpose.

<details><summary>💡 Hint 1 — what to think about</summary>

Which spread are the limits meant to describe: how precisely you know the baseline rate, or
how far ONE window of `window_size` decisions wanders when nothing is wrong? That decides
which count goes under the square root. Then ask what a limit outside the range a rate can
take would mean on a chart, and what limits fitted to no baseline at all ought to let
through.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Deal with the degenerate case first: if either count is not positive, return the permissive
limits the docstring lists before anything divides. Otherwise the centre is the baseline's
error share, sigma is the binomial standard error over the WINDOW size, and each limit is the
centre plus or minus `k` sigmas, clamped into the unit interval only where it escapes. Echo
`k`, `window_size` and `baseline_n` back rather than a constant.
</details>

In [ ]:
def control_limits(baseline_errors: int, baseline_n: int, window_size: int,
                   k: float = DEFAULT_K) -> dict:
    """p-chart control limits for a rate measured over windows of `window_size` decisions.

    centre = baseline_errors / baseline_n
    sigma  = sqrt(centre * (1 - centre) / window_size)   <- window_size, not baseline_n
    ucl    = centre + k * sigma, capped at 1.0
    lcl    = centre - k * sigma, floored at 0.0

    A rate cannot leave [0, 1], so a chart whose upper limit is 1.07 is telling you it can
    never alert. Cap it and say so rather than plotting an impossible number.

    When `baseline_n` or `window_size` is zero or negative, return centre 0.0, sigma 0.0,
    ucl 1.0, lcl 0.0: a baseline of nothing constrains nothing, and limits of (0.0, 0.0)
    there would page on the first window you ever plot.

    Returns a dict with keys "centre", "sigma", "k", "ucl", "lcl", "window_size",
    "baseline_n".

    Example:
        >>> lim = control_limits(48, 800, 50, 3.0)
        >>> round(lim["centre"], 4), round(lim["sigma"], 6), round(lim["ucl"], 6)
        (0.06, 0.033586, 0.160757)
        >>> control_limits(0, 0, 50)["ucl"]
        1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_control_limits() -> None:
    lim = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, DEFAULT_K)
    assert set(lim) == {"centre", "sigma", "k", "ucl", "lcl", "window_size", "baseline_n"}, (
        f"keys were {sorted(lim)} — return exactly the seven documented names"
    )
    assert abs(lim["centre"] - 0.06) < 1e-12, (
        f"centre is {lim['centre']!r}; the baseline is {BASELINE_ERRORS} errors in "
        f"{BASELINE_N} decisions, which is exactly 0.06"
    )
    want_sigma = math.sqrt(0.06 * 0.94 / WINDOW_SIZE)
    assert abs(lim["sigma"] - want_sigma) < 1e-12, (
        f"sigma is {lim['sigma']:.6f}, expected {want_sigma:.6f} — the standard error is "
        "sqrt(p*(1-p)/WINDOW_SIZE). Dividing by baseline_n instead gives limits that describe "
        "the spread of the baseline mean, not of the windows you are about to plot"
    )
    assert abs(lim["ucl"] - (0.06 + DEFAULT_K * want_sigma)) < 1e-12, (
        "ucl is centre + k*sigma"
    )
    assert lim["lcl"] == 0.0, (
        f"lcl is {lim['lcl']!r}; 0.06 - 3*sigma is negative and a rate cannot be, so floor "
        "it at 0.0 rather than plotting an impossible limit"
    )
    wide = control_limits(400, 800, 4, 3.0)
    assert wide["ucl"] == 1.0, (
        f"with p = 0.5 over windows of 4, centre + 3*sigma is {0.5 + 3 * math.sqrt(0.25 / 4):.2f}"
        " — cap the upper limit at 1.0"
    )
    empty = control_limits(0, 0, WINDOW_SIZE)
    assert (empty["centre"], empty["sigma"], empty["ucl"], empty["lcl"]) == (0.0, 0.0, 1.0, 0.0), (
        f"an empty baseline returned {empty} — a baseline of nothing constrains nothing, so "
        "centre 0.0, sigma 0.0, ucl 1.0, lcl 0.0. Returning ucl 0.0 pages on every window"
    )
    tight = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, 1.0)
    assert tight["ucl"] < lim["ucl"], "a smaller k must give a tighter upper limit"
    print(f"exercise 1 looks right — centre {lim['centre']:.4f}, sigma {lim['sigma']:.6f}, "
          f"ucl {lim['ucl']:.6f}")

In [ ]:
_try("exercise 1", _check_control_limits)

## 4. Exercise 2 — the windowed monitor

Now the monitor itself. Cut the feed into **tumbling** windows of `WINDOW_SIZE` decisions,
score each one against the limits, and raise an alert when the excursion is confirmed.
Three decisions are yours to get right, and each is a way real monitors fail:

- **Cut on the sequence number, never on the timestamp.** P01-L01 taught this with one
  backwards clock; this feed has a whole batch a broken exporter flushed days late — the
  chart cell below counts the batch and says how late. Sort by `ts` and every row in it
  lands in a window it does not belong to, along with everything it displaces.
- **Drop the incomplete tail.** The last few decisions are not a window. Their rate is a
  statement about where you stopped counting.
- **Confirm before paging.** A single window outside the limits is an excursion; `MIN_RUN`
  consecutive ones is a signal. The alert is reported at the *first* window of the run, so
  the record says when the process changed, not when you noticed.

<details><summary>💡 Hint 1 — what to think about</summary>

Three questions decide this function. Which field is the authority on order when a batch of
rows carries timestamps days late? Is the handful of rows left over at the end a short
window, or nothing? And when several outside windows follow one another, how many alerts is
that, and which window does the record name?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sort by `seq` (a sorted copy, not the caller's list) and take only as many whole windows as
fit; the remainder is `dropped_rows`. Mark a window outside only when its rate is strictly
above `ucl` or strictly below `lcl`. Then walk the windows keeping a run length: add one on
an outside window, reset it on an inside one, and record one alert at the moment the run
first reaches `min_run`, naming the window where that run began.
</details>

In [ ]:
def windowed_monitor(rows: list, limits: dict, window_size: int = WINDOW_SIZE,
                     min_run: int = MIN_RUN) -> dict:
    """Score a feed in tumbling windows and raise confirmed alerts.

    Process `rows` in ascending `row["seq"]` order. Take non-overlapping windows of
    `window_size` rows; any rows left over at the end are NOT a window — count them in
    "dropped_rows" and stop.

    Each window is a dict:
      "index"        0-based window number
      "n"            window_size
      "errors"       rows with a truthy "wrong"
      "rate"         errors / n
      "first_seq", "last_seq"   the seq of the first and last row in the window
      "first_ts", "last_ts"     their timestamps
      "outside"      rate > limits["ucl"] or rate < limits["lcl"], STRICTLY outside, so a
                     window that lands exactly on a limit is inside it

    "alerts" is the list of window indices at which a confirmed excursion STARTS: for every
    maximal run of consecutive "outside" windows whose length is at least `min_run`, the
    index of the first window of that run. One alert per run, not one per window.

    Returns {"n_windows", "dropped_rows", "windows", "alerts", "verdict"}, where the verdict
    is "alert" when there is any alert and "in_control" otherwise.

    Example:
        >>> lim = {"ucl": 0.1, "lcl": 0.0}
        >>> rows = [{"seq": i, "ts": "2026-01-01T00:00:00Z", "wrong": i % 2 == 0}
        ...         for i in range(4)]
        >>> rep = windowed_monitor(rows, lim, window_size=2, min_run=2)
        >>> rep["n_windows"], rep["dropped_rows"], rep["alerts"]
        (2, 0, [0])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_monitor() -> None:
    lim = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, DEFAULT_K)
    rep = windowed_monitor(PROVIDER_FEED, lim)
    assert rep["n_windows"] == len(WINDOW_ERRORS), (
        f"n_windows is {rep['n_windows']}, expected {len(WINDOW_ERRORS)} — "
        f"{len(PROVIDER_FEED)} rows is {len(WINDOW_ERRORS)} whole windows of {WINDOW_SIZE} "
        f"plus {TAIL_ROWS} left over, and the leftovers are not a window"
    )
    assert rep["dropped_rows"] == TAIL_ROWS, (
        f"dropped_rows is {rep['dropped_rows']}, expected {TAIL_ROWS}"
    )
    assert [w["errors"] for w in rep["windows"]] == list(WINDOW_ERRORS), (
        "the per-window error counts do not match the counts the fixture was built with. The "
        "usual cause is sorting by 'ts': thirty decisions in this feed were flushed ten days "
        "late, and ordering by the clock moves them into the wrong windows"
    )
    assert rep["windows"][0]["first_seq"] == 0 and rep["windows"][0]["last_seq"] == 49, (
        "window 0 covers seq 0..49; report the first and last seq so an inspector can find the"
        " decisions behind a breach"
    )
    assert rep["windows"][28]["outside"] is True, (
        f"window 28 has a rate of {WINDOW_ERRORS[28] / WINDOW_SIZE:.3f} against an upper limit "
        f"of {lim['ucl']:.3f}; it is outside"
    )
    assert rep["windows"][27]["outside"] is False, (
        f"window 27 has a rate of {WINDOW_ERRORS[27] / WINDOW_SIZE:.3f}, inside an upper limit "
        f"of {lim['ucl']:.3f}. It is degraded and this chart cannot see it — which is the "
        "whole subject of the next exercise, so do not flag it by widening the comparison"
    )
    assert rep["alerts"] == [28], (
        f"alerts came out {rep['alerts']}, expected [28]. At k={DEFAULT_K} exactly windows 28, "
        "29 and 30 are outside; they are consecutive, so they are ONE confirmed alert, "
        "reported at the first window of the run"
    )
    assert rep["verdict"] == "alert", "there is an alert, so the verdict is 'alert'"

    edge = [{"seq": i, "ts": iso(AS_OF_DT), "wrong": i < 1} for i in range(4)]
    on_limit = windowed_monitor(edge, {"ucl": 0.5, "lcl": 0.0}, window_size=2, min_run=1)
    assert on_limit["alerts"] == [], (
        "a window whose rate lands exactly ON the upper limit is inside it. Use a strict "
        "comparison, or every chart you draw alerts on its own centre line eventually"
    )
    gap = [{"seq": i, "ts": iso(AS_OF_DT), "wrong": i in (0, 4)} for i in range(6)]
    lone = windowed_monitor(gap, {"ucl": 0.4, "lcl": 0.0}, window_size=2, min_run=2)
    assert lone["alerts"] == [], (
        "two separate single-window excursions are not a confirmed run of two. Reset the run "
        "counter on every window that is inside the limits"
    )
    print(f"exercise 2 looks right — {rep['n_windows']} windows, {rep['dropped_rows']} rows "
          f"dropped, alerts {rep['alerts']}")

In [ ]:
_try("exercise 2", _check_monitor)

The next cell prints the chart and what the two rejected shortcuts would have done: the
incomplete tail scored as if it were a window, and the whole feed ordered by timestamp.

In [ ]:
# The chart and the refit demo both print your monitor's windows.
_FOR_CHART = ("exercise 1", "exercise 2")


def _show_chart() -> None:
    lim = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, DEFAULT_K)
    rep = windowed_monitor(PROVIDER_FEED, lim)
    print(f"centre {lim['centre']:.4f}  ucl {lim['ucl']:.4f}  (k={lim['k']}, "
          f"window {WINDOW_SIZE})\n")
    for w in rep["windows"]:
        bar = "#" * w["errors"]
        mark = "  <-- outside" if w["outside"] else ""
        print(f"  w{w['index']:02d} seq {w['first_seq']:4d}-{w['last_seq']:4d} "
              f"{w['errors']:3d}/{w['n']} {w['rate']:.3f} {bar:14s}{mark}")
    print(f"\nalerts at {rep['alerts']}")
    tail_rate = TAIL_ERRORS / TAIL_ROWS
    print(f"\nthe {TAIL_ROWS} dropped rows hold {TAIL_ERRORS} errors, a rate of "
          f"{tail_rate:.3f} — {tail_rate / lim['ucl']:.1f} times the upper limit. They are "
          f"{TAIL_ROWS} decisions, not a window.")
    # How big is the late batch, and how late? MEASURED against the feed's own nominal
    # cadence, rather than read off the constant that planted it — the same thing you would
    # do to a real feed, where no constant exists to read.
    n_rows = len(PROVIDER_FEED)
    lateness = [(row["seq"],
                 (parse_ts(row["ts"]) - (AS_OF_DT - timedelta(
                     hours=DECISION_INTERVAL_HOURS * (n_rows - 1 - row["seq"])))
                  ).total_seconds())
                for row in PROVIDER_FEED]
    flushed = [(seq, late) for seq, late in lateness if late > 3600]
    print(f"\n{len(flushed)} rows sit off this feed's own {DECISION_INTERVAL_HOURS}-hour "
          f"cadence, seq {flushed[0][0]}..{flushed[-1][0]}, the latest by "
          f"{max(late for _, late in flushed) / 86400:.1f} days. Their sequence numbers are "
          "still right; only their clock is wrong.")
    by_ts = [dict(r, seq=i) for i, r in enumerate(
        sorted(PROVIDER_FEED, key=lambda r: (r["ts"], r["seq"])))]
    clocked = windowed_monitor(by_ts, lim)
    moved = [i for i, (a, b) in enumerate(zip(rep["windows"], clocked["windows"]))
             if a["errors"] != b["errors"]]
    print(f"ordered by timestamp instead of sequence, {len(moved)} windows change their error "
          f"count: {moved}. At k={DEFAULT_K} the alert happens to survive that.")


_try("chart", _show_chart, needs=_FOR_CHART)

### The defect the capstone plants

Now that you have a monitor, refit the limits the tempting way: over the in-service period,
the one that contains the degradation. The cell below does that and re-runs the same
monitor against the result.

In [ ]:
def _show_refit() -> None:
    honest = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, DEFAULT_K)
    after_errors = sum(WINDOW_ERRORS[BASELINE_WINDOWS:])
    after_n = (len(WINDOW_ERRORS) - BASELINE_WINDOWS) * WINDOW_SIZE
    refit = control_limits(after_errors, after_n, WINDOW_SIZE, DEFAULT_K)
    worst = max(WINDOW_ERRORS) / WINDOW_SIZE
    honest_alerts = windowed_monitor(PROVIDER_FEED, honest)["alerts"]
    refit_alerts = windowed_monitor(PROVIDER_FEED, refit)["alerts"]
    print(f"baseline-fitted : centre {honest['centre']:.4f}  ucl {honest['ucl']:.4f}  "
          f"alerts {honest_alerts}")
    print(f"refitted after  : centre {refit['centre']:.4f}  ucl {refit['ucl']:.4f}  "
          f"alerts {refit_alerts}")
    print(f"the widest window in the whole feed has a rate of {worst:.4f}")
    print(f"\nthe refit moves the upper limit up by a factor of "
          f"{refit['ucl'] / honest['ucl']:.2f} and the alerts from {honest_alerts} to "
          f"{refit_alerts}.")
    print("Nothing was falsified to do that. The limits were simply computed from data that")
    print("included the thing they were supposed to find.")


_try("refit demo", _show_refit, needs=_FOR_CHART)

## 5. Exercise 3 — what a mis-tuned monitor costs, in both directions

`k` is the only dial most monitors have, and it is almost always set by habit. Here you
price it. On this fixture — and only because it is a fixture — we know which windows were
genuinely degraded, so both kinds of error can be counted:

- a **false alert** is a confirmed alert that starts outside the degraded windows. It costs
  `COST_PER_FALSE_ALERT`, and it costs something else that is not money: after
  `SWITCH_OFF_AFTER` of them the team mutes the monitor. From that point it raises nothing
  and sees nothing, which is the real price of a jumpy chart.
- a **missed window** is a degraded window that no live alert ever covered — because the
  monitor was too wide to notice, or because it had already been muted. It costs
  `COST_PER_MISSED_WINDOW`.

Both constants are this lesson's, and the number that matters is their ratio, not either
one. The sweep cell after the exercise shows the optimum moving when you change it.

<details><summary>💡 Hint 1 — what to think about</summary>

A muted monitor is not a quieter monitor. From the alert that mutes it onward nothing it
would have raised counts, true or false, so the order you walk the alerts in matters and the
walk can end early. Then: which degraded windows did no LIVE alert cover — before the first
genuine detection, and after the mute? And what is the delay of a monitor that never detected
at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the alerts in ascending order, keeping each as live and counting the ones not in
`degraded`; when that count reaches `switch_off_after`, record the alert as `switched_off_at`
and stop. The first detection is the smallest live alert that IS degraded, or None. A
degraded window is missed when there is no detection, when it precedes the detection, or when
it follows the mute. With no detection the delay is None, not zero; the costs are counts ×
prices.
</details>

In [ ]:
def monitor_costs(report: dict, degraded, cost_false: int = COST_PER_FALSE_ALERT,
                  cost_missed: int = COST_PER_MISSED_WINDOW,
                  switch_off_after: int = SWITCH_OFF_AFTER) -> dict:
    """Price a monitor run against the windows that were genuinely degraded.

    Walk `report["alerts"]` in ascending order. Each alert is LIVE until the monitor is muted.
    An alert whose index is in `degraded` is true; any other is false. Count the false ones as
    you go, and when that count reaches `switch_off_after`, record that alert's index as
    "switched_off_at" and treat every LATER alert as though it never fired.

    Then:
      "false_alerts"   live alerts not in `degraded`
      "true_alerts"    live alerts in `degraded`
      "first_detection"  the smallest live true alert index, or None
      "missed_windows"   sorted degraded windows that no live alert covered: every degraded
                         window when "first_detection" is None, otherwise those before it,
                         plus those after "switched_off_at" when the monitor was muted
      "detection_delay_windows"  first_detection - min(degraded), or None
      "false_alert_cost", "missed_cost", "total_cost"

    Returns those nine keys plus "switched_off_at".

    Example:
        >>> rep = {"alerts": [4, 9]}
        >>> c = monitor_costs(rep, {9, 10}, switch_off_after=3)
        >>> c["false_alerts"], c["missed_windows"], c["total_cost"]
        (1, [], 400)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_costs() -> None:
    lim = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, DEFAULT_K)
    rep = windowed_monitor(PROVIDER_FEED, lim)
    cost = monitor_costs(rep, DEGRADED_WINDOWS)
    assert cost["false_alerts"] == 0 and cost["true_alerts"] == 1, (
        f"at k={DEFAULT_K} the single alert is inside the degraded windows, so 0 false and "
        f"1 true; you reported {cost['false_alerts']} and {cost['true_alerts']}"
    )
    assert cost["first_detection"] == 28 and cost["detection_delay_windows"] == 2, (
        f"detection is at window 28 and the degradation started at {min(DEGRADED_WINDOWS)}, "
        f"so the delay is 2 windows; you reported {cost['detection_delay_windows']}"
    )
    assert cost["missed_windows"] == [26, 27], (
        f"windows 26 and 27 degraded and no alert covered them; you reported "
        f"{cost['missed_windows']}"
    )
    assert cost["total_cost"] == 2 * COST_PER_MISSED_WINDOW, (
        f"two missed windows and no false alerts is {2 * COST_PER_MISSED_WINDOW}; you reported "
        f"{cost['total_cost']}"
    )

    jumpy = windowed_monitor(PROVIDER_FEED,
                             control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, 1.0))
    muted = monitor_costs(jumpy, DEGRADED_WINDOWS)
    assert muted["switched_off_at"] == 22, (
        f"at k=1.0 the alerts are {jumpy['alerts']}: three of them fire before the "
        f"degradation, and the third mutes the monitor at window 22. You reported "
        f"switched_off_at={muted['switched_off_at']}"
    )
    assert muted["false_alerts"] == SWITCH_OFF_AFTER, (
        "stop counting alerts once the monitor is muted — a muted monitor cannot page you a "
        "fourth time"
    )
    assert muted["missed_windows"] == list(DEGRADED_WINDOWS), (
        f"the monitor was muted at window 22 and every degraded window comes later, so all "
        f"{len(DEGRADED_WINDOWS)} are missed; you reported {muted['missed_windows']}"
    )
    blind = monitor_costs({"alerts": []}, DEGRADED_WINDOWS)
    assert blind["first_detection"] is None and blind["detection_delay_windows"] is None, (
        "a monitor that never alerts has no detection and no delay — None, not 0"
    )
    assert blind["missed_windows"] == list(DEGRADED_WINDOWS), (
        "a monitor that never alerts misses everything"
    )
    print(f"exercise 3 looks right — at k={DEFAULT_K} the bill is {cost['total_cost']}, at "
          f"k=1.0 it is {muted['total_cost']}")

In [ ]:
_try("exercise 3", _check_costs)

Now sweep `k` and read the bill. The first column is the dial; the last is what it costs.

In [ ]:
# The sweep prices every k with your monitor AND your cost function.
_FOR_SWEEP = ("exercise 1", "exercise 2", "exercise 3")


def _show_sweep() -> None:
    rows = []
    for k in K_GRID:
        lim = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, k)
        rep = windowed_monitor(PROVIDER_FEED, lim)
        rows.append((k, lim["ucl"], rep["alerts"], monitor_costs(rep, DEGRADED_WINDOWS)))
    print(f"{'k':>4s} {'ucl':>7s} {'alerts':>14s} {'false':>6s} {'missed':>7s} "
          f"{'muted':>6s} {'delay':>6s} {'cost':>7s}")
    for k, ucl, alerts, c in rows:
        muted = "-" if c["switched_off_at"] is None else f"w{c['switched_off_at']}"
        delay = "-" if c["detection_delay_windows"] is None else c["detection_delay_windows"]
        print(f"{k:>4.1f} {ucl:>7.4f} {str(alerts):>14s} {c['false_alerts']:>6d} "
              f"{len(c['missed_windows']):>7d} {muted:>6s} {str(delay):>6s} "
              f"{c['total_cost']:>7d}")
    best = min(rows, key=lambda r: (r[3]["total_cost"], r[0]))
    tightest, widest = rows[0], rows[-1]
    textbook = [r for r in rows if r[0] == DEFAULT_K][0]
    print(f"\ncheapest k on this feed: {best[0]} at {best[3]['total_cost']}")
    print(f"the textbook k={DEFAULT_K} costs {textbook[3]['total_cost']}, which is "
          f"{textbook[3]['total_cost'] - best[3]['total_cost']} more, for "
          f"{len(textbook[3]['missed_windows']) - len(best[3]['missed_windows'])} extra "
          "missed windows. It is a default, not an answer.")
    print(f"k={tightest[0]}: {tightest[3]['false_alerts']} false pages mute it at window "
          f"{tightest[3]['switched_off_at']}, so it then misses "
          f"{len(tightest[3]['missed_windows'])} degraded windows and bills "
          f"{tightest[3]['total_cost']}.")
    print(f"k={widest[0]}: {len(widest[2])} alerts ever, "
          f"{len(widest[3]['missed_windows'])} degraded windows missed, bill "
          f"{widest[3]['total_cost']} — the same bill as having no monitor at all.")


_try("k sweep", _show_sweep, needs=_FOR_SWEEP)

In [ ]:
def _show_ratio_moves_the_answer() -> None:
    """The optimum is a property of the cost RATIO, not of the chart. Measure that."""
    print(f"{'a page costs':>13s} {'a miss costs':>13s} {'best k':>7s} {'cost':>7s}")
    chosen = []
    for cost_false in (400, 2000, 6000, 20000):
        scored = []
        for k in K_GRID:
            rep = windowed_monitor(
                PROVIDER_FEED, control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, k))
            scored.append((k, monitor_costs(rep, DEGRADED_WINDOWS,
                                            cost_false=cost_false)["total_cost"]))
        best_k, best_cost = min(scored, key=lambda r: (r[1], r[0]))
        chosen.append(best_k)
        print(f"{cost_false:>13d} {COST_PER_MISSED_WINDOW:>13d} {best_k:>7.1f} "
              f"{best_cost:>7d}")
    print(f"\nSame feed, same chart, {len(set(chosen))} different answers: "
          f"{sorted(set(chosen))}. A monitoring plan that states k without stating what a")
    print("page and a missed window cost has not made a decision; it has copied one.")


_try("cost ratio", _show_ratio_moves_the_answer, needs=_FOR_SWEEP)

## 6. Exercise 4 — the deployer's clock

Article 72(2) says the data may be provided by deployers, and Article 26(5) makes the
deployer send it. What arrives is a feed on somebody else's clock. This one reads local
time and is labelled UTC, so every row is shifted by the same amount — except a handful that
were re-exported after somebody corrected a date by hand.

Estimate the offset from the rows both parties hold: match on `decision_id`, take the
difference, and use the **median**. The mean is the obvious choice and it is wrong here,
because a handful of re-exported rows drag it. Report the residual spread as well, because
an offset with no spread beside it is a number you cannot act on: it is the spread that
tells you whether the merged order is trustworthy to seconds or to hours.

<details><summary>💡 Hint 1 — what to think about</summary>

Which way round is the difference? The offset is how far the DEPLOYER's clock runs ahead, so
a positive number has to mean the deployer's timestamp is the later one. Then ask what a
handful of rows re-exported a whole day late do to the mean, the median and the standard
deviation of otherwise tightly bunched deltas. The centre and the spread you report must
both shrug them off.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Index the provider's timestamps by `decision_id`, and for each deployer row that matches take
deployer time minus provider time, in seconds, parsing both. If nothing matched, return the
all-zero result before any statistics run. Otherwise the offset is the median delta, the
spread is the median of each delta's absolute distance from that median, and an outlier is a
pair further from the median than the `tolerance` ARGUMENT — not the module constant.
</details>

In [ ]:
def estimate_clock_offset(provider: list, deployer: list,
                          tolerance: float = OUTLIER_TOLERANCE_SECONDS) -> dict:
    """Estimate how far the deployer's clock runs ahead of the provider's.

    Pair rows by "decision_id". For each pair the delta is
    `(deployer ts - provider ts)` in seconds, positive when the deployer's clock is ahead.

    Returns:
      "n_pairs"          how many decisions both parties hold
      "offset_seconds"   the MEDIAN delta, as a float
      "mean_seconds"     the mean delta, reported so you can see what it would have cost
      "spread_seconds"   the median of |delta - median|, the median absolute deviation
      "outliers"         pairs whose |delta - median| exceeds `tolerance`

    With no pairs at all return n_pairs 0 and every number 0.0 — an unmatched feed gives you
    no offset, and guessing zero offset silently is how a two-hour error becomes permanent.

    Example:
        >>> p = [{"decision_id": "D-1", "ts": "2026-01-01T00:00:00Z"},
        ...      {"decision_id": "D-2", "ts": "2026-01-01T01:00:00Z"}]
        >>> d = [{"decision_id": "D-1", "ts": "2026-01-01T00:00:10Z"},
        ...      {"decision_id": "D-2", "ts": "2026-01-01T01:00:20Z"}]
        >>> est = estimate_clock_offset(p, d)
        >>> est["n_pairs"], est["offset_seconds"]
        (2, 15.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_offset() -> None:
    est = estimate_clock_offset(PROVIDER_FEED, DEPLOYER_FEED)
    assert est["n_pairs"] == 100, (
        f"n_pairs is {est['n_pairs']}, expected 100 — the deployer sends 160 rows, and 100 of "
        "them are decisions the provider also logged. Pair on decision_id"
    )
    assert abs(est["offset_seconds"] - 7620.0) < 1e-9, (
        f"offset_seconds is {est['offset_seconds']:.1f}, expected 7620.0 — the median delta. "
        "Subtract the PROVIDER's timestamp from the DEPLOYER's, not the other way round"
    )
    assert est["mean_seconds"] > est["offset_seconds"] + 1500, (
        f"mean_seconds is {est['mean_seconds']:.1f} and should be far above the median: a "
        "handful of re-exported rows drag it. If yours equals the median, you are reporting "
        "the median twice"
    )
    assert est["spread_seconds"] < 10, (
        f"spread_seconds is {est['spread_seconds']:.1f}; the median absolute deviation here "
        "is a few seconds. Using the standard deviation instead lets the same four outliers "
        "that broke the mean break the spread as well"
    )
    assert est["outliers"] == 4, (
        f"outliers is {est['outliers']}, expected 4 — pairs more than "
        f"{OUTLIER_TOLERANCE_SECONDS}s from the median"
    )
    none = estimate_clock_offset(PROVIDER_FEED, [{"decision_id": "X-1", "ts": iso(AS_OF_DT)}])
    assert none["n_pairs"] == 0 and none["offset_seconds"] == 0.0, (
        "a feed with nothing in common gives n_pairs 0 and offset 0.0, not a crash"
    )
    print(f"exercise 4 looks right — median {est['offset_seconds']:.0f}s, mean "
          f"{est['mean_seconds']:.0f}s, spread {est['spread_seconds']:.1f}s, "
          f"{est['outliers']} outliers")

In [ ]:
_try("exercise 4", _check_offset)

In [ ]:
def _show_clock() -> None:
    est = estimate_clock_offset(PROVIDER_FEED, DEPLOYER_FEED)
    drift = est["mean_seconds"] - est["offset_seconds"]
    print(f"median offset {est['offset_seconds']:.0f}s = "
          f"{est['offset_seconds'] / 3600:.2f} h")
    print(f"mean   offset {est['mean_seconds']:.0f}s = {est['mean_seconds'] / 3600:.2f} h")
    print(f"using the mean would misplace every deployer row by {drift:.0f}s "
          f"({drift / 60:.0f} minutes), from {est['outliers']} bad rows out of "
          f"{est['n_pairs']}.")
    print(f"half the pairs sit within {est['spread_seconds']:.1f}s of the median, so the "
          "merged order below is trustworthy to a few seconds — which is worth saying out")
    print("loud, because the same statistic on a feed with a drifting clock would not be.")


_try("clock demo", _show_clock, needs=("exercise 4",))

## 7. Exercise 5 — merging the deployer feed without counting it twice

Now fold the feed in. The deployer holds two kinds of row: decisions you already have, and
decisions you do not — the ones its edge deployment took while it could not reach you. The
second kind is the entire point of Article 72(2). The first kind is a trap: append it and
your denominator grows by rows you already counted, and the rate you report is diluted by
exactly the evidence you thought you were adding.

One honest tension. Inside one log, `seq` is the authority and the clock is not. Across two
logs there is no shared `seq`, so the merged order has to fall back on the reconciled clock
— which is why you measured its spread first. Report `out_of_seq_rows`, the number of your
own rows the merge has moved out of sequence order, so the price of that fallback is on the
face of the artefact rather than in somebody's head.

<details><summary>💡 Hint 1 — what to think about</summary>

Two kinds of deployer row arrive, and only one kind is evidence you did not already have. For
the rows you do add, think about the direction of the correction: the offset says how far the
deployer's clock runs AHEAD, so which way must a deployer timestamp move to read on yours?
And once everything is ordered by the reconciled clock, how would you notice your own rows
leaving their `seq` order?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start the merged list with every provider row (`src_seq` = its own `seq`). For each deployer
row, count and skip it if its `decision_id` is already held; otherwise move its parsed
timestamp BACK by `offset_seconds`, format it with `iso()`, and add it with `src_seq` None.
Sort on `(ts, decision_id)` and renumber `seq` from zero. For `out_of_seq_rows`, compare the
provider rows' `src_seq` in merged order with the same values sorted, position by position.
The error rate divides by every merged row, not by the added ones.
</details>

In [ ]:
def merge_deployer_feed(provider: list, deployer: list, offset_seconds: float) -> dict:
    """Merge a deployer feed into the provider's own view, on a common clock.

    A deployer row whose "decision_id" the provider already holds is an OVERLAP: count it and
    drop it. Every other deployer row is ADDED, with its timestamp corrected by subtracting
    `offset_seconds` so it reads on the provider's clock.

    Every merged row is {"seq", "ts", "decision_id", "wrong", "source", "src_seq"}, where
    "source" is "provider" or "deployer" and "src_seq" is the provider's own sequence number
    or None. Sort the merged rows by ("ts", "decision_id") and number them 0..M-1 in "seq".

    Returns {"merged", "n_provider", "n_deployer", "n_overlap", "n_added", "out_of_seq_rows",
    "offset_seconds", "merged_error_rate"}, where "out_of_seq_rows" compares the provider rows
    in merged order against the same rows in src_seq order, position by position, and counts
    the positions that disagree; and "merged_error_rate" is the share of ALL merged rows whose
    "wrong" is truthy.

    Example:
        >>> p = [{"seq": 0, "decision_id": "D-0", "ts": "2026-01-01T00:00:00Z",
        ...       "wrong": False}]
        >>> d = [{"decision_id": "D-0", "ts": "2026-01-01T00:00:10Z", "wrong": False},
        ...      {"decision_id": "P-0", "ts": "2026-01-01T00:00:20Z", "wrong": True}]
        >>> rep = merge_deployer_feed(p, d, 10.0)
        >>> rep["n_overlap"], rep["n_added"], rep["merged_error_rate"]
        (1, 1, 0.5)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_merge() -> None:
    est = estimate_clock_offset(PROVIDER_FEED, DEPLOYER_FEED)
    rep = merge_deployer_feed(PROVIDER_FEED, DEPLOYER_FEED, est["offset_seconds"])
    assert rep["n_overlap"] == 100 and rep["n_added"] == 60, (
        f"overlap/added came out {rep['n_overlap']}/{rep['n_added']}, expected 100/60 — the "
        "overlap is dropped, not appended"
    )
    assert len(rep["merged"]) == len(PROVIDER_FEED) + 60, (
        f"the merged feed has {len(rep['merged'])} rows; it should be every provider row plus "
        f"the {rep['n_added']} the deployer added. Appending the whole deployer feed inflates "
        f"it by {rep['n_overlap']} rows you already had"
    )
    assert [r["seq"] for r in rep["merged"][:3]] == [0, 1, 2], "renumber seq from 0"
    stamps = [r["ts"] for r in rep["merged"]]
    assert stamps == sorted(stamps), "the merged rows are ordered by corrected timestamp"
    added = [r for r in rep["merged"] if r["source"] == "deployer"]
    assert all(r["src_seq"] is None for r in added), (
        "an added deployer row has no provider sequence number; src_seq is None"
    )
    first_added = parse_ts(added[0]["ts"])
    raw = min(parse_ts(r["ts"]) for r in DEPLOYER_FEED if r["decision_id"].startswith("P-"))
    assert abs((raw - first_added).total_seconds() - est["offset_seconds"]) < 1, (
        "correct an added row's timestamp by SUBTRACTING the offset; adding it puts the "
        "deployer's rows more than four hours from where they belong"
    )
    assert rep["out_of_seq_rows"] == 81, (
        f"out_of_seq_rows is {rep['out_of_seq_rows']}, expected 81 — the thirty decisions a "
        "broken exporter flushed late, plus everything they jump over. Compare the provider "
        "rows in merged order against the same rows sorted by src_seq, position by position"
    )
    errors = sum(1 for r in rep["merged"] if r["wrong"])
    assert abs(rep["merged_error_rate"] - errors / len(rep["merged"])) < 1e-12, (
        "merged_error_rate is over ALL merged rows, not over the added ones"
    )
    print(f"exercise 5 looks right — {rep['n_overlap']} overlapping, {rep['n_added']} added, "
          f"{len(rep['merged'])} rows, {rep['out_of_seq_rows']} out of sequence")

In [ ]:
_try("exercise 5", _check_merge)

Run the monitor again on the merged feed. The deployer's rows are the only place one of
these failures is visible at all.

In [ ]:
# The ingest demo runs your monitor over your merge, on your clock offset.
_FOR_INGEST = ("exercise 1", "exercise 2", "exercise 4", "exercise 5")


def _show_ingest() -> None:
    lim = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, DEFAULT_K)
    est = estimate_clock_offset(PROVIDER_FEED, DEPLOYER_FEED)
    own = windowed_monitor(PROVIDER_FEED, lim)
    rep = merge_deployer_feed(PROVIDER_FEED, DEPLOYER_FEED, est["offset_seconds"])
    both = windowed_monitor(rep["merged"], lim)
    print(f"provider feed alone : {own['n_windows']} windows, alerts {own['alerts']}")
    print(f"with the deployer   : {both['n_windows']} windows, alerts {both['alerts']}")
    new = [a for a in both["alerts"] if a not in own["alerts"]]
    print(f"\n{len(new)} alert(s) exist only once the deployer's rows are in: {new}")
    for a in new:
        w = both["windows"][a]
        print(f"  window {a}: {w['errors']}/{w['n']} = {w['rate']:.3f} against an upper limit "
              f"of {lim['ucl']:.3f}, from {w['first_ts']}")
    # what appending the overlap instead of dropping it does to the same windows
    naive = [dict(r) for r in rep["merged"]]
    have = {r["decision_id"] for r in PROVIDER_FEED}
    for r in DEPLOYER_FEED:
        if r["decision_id"] in have:
            naive.append({"ts": iso(parse_ts(r["ts"])
                                    - timedelta(seconds=est["offset_seconds"])),
                          "decision_id": r["decision_id"], "wrong": r["wrong"],
                          "source": "deployer", "src_seq": None})
    naive.sort(key=lambda r: (r["ts"], r["decision_id"]))
    for i, r in enumerate(naive):
        r["seq"] = i
    dbl = windowed_monitor(naive, lim)
    tail_true = [w for w in both["windows"] if w["index"] >= min(new)] if new else []
    tail_dbl = [w for w in dbl["windows"] if w["index"] >= min(new)] if new else []
    rate_true = sum(w["errors"] for w in tail_true) / max(1, sum(w["n"] for w in tail_true))
    rate_dbl = sum(w["errors"] for w in tail_dbl) / max(1, sum(w["n"] for w in tail_dbl))
    print(f"\nover the windows from {min(new) if new else 0} on, the error rate is "
          f"{rate_true:.4f} merged correctly and {rate_dbl:.4f} with the overlap appended "
          f"twice: understated by {100 * (1 - rate_dbl / rate_true):.0f}%.")
    print(f"The double count also adds {len(naive) - len(rep['merged'])} rows of evidence "
          "that is not evidence, because you already had it.")


_try("ingest", _show_ingest, needs=_FOR_INGEST)

## 8. Exercise 6 — classifying a serious incident

A drifting metric is not an incident, and an incident is not a drifting metric. The monitor
above tells you something changed. Article 73 asks a different question with a different
clock on it: is this an incident or malfunctioning that *directly or indirectly leads to*
one of the four harms in Article 3(49)?

The branch order matters. The two-day deadline in 73(3) is triggered by a **widespread
infringement** or by point **(b)** — critical infrastructure. The ten-day deadline in 73(4)
is triggered by **death**. Everything else serious is 73(2)'s fifteen days.

One case the Regulation does not resolve: an incident that is both a death and a critical
infrastructure disruption. 73(3) and 73(4) each say "notwithstanding paragraph 2", and
neither says what happens when both apply. **This lesson takes the shorter deadline**,
because a tie-break that resolves an overlap by choosing the later date converts an overlap
into a delay. That is the lesson's choice, not the Regulation's, and it is in `claims.yaml`
as one.

<details><summary>💡 Hint 1 — what to think about</summary>

Two instincts are wrong here. A fundamental-rights infringement feels like the urgent case,
and any point (a) harm feels like the death clock. Read which TWO conditions trigger 73(3),
and which single fact triggers 73(4). Then ask what your branch order does to an incident
that is both a death and a critical-infrastructure disruption.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sort the points into a tuple first; with none, it is not serious and has no paragraph and no
clock. Otherwise test the short-deadline branch — the `widespread` flag, or point (b) among
the points — BEFORE the `death` flag, and let everything else fall through to 73(2). Read the
days from `DEADLINE_DAYS` by paragraph rather than typing them, and write a trigger that
names what decided the branch in words a reviewer could argue with.
</details>

In [ ]:
def _inc(incident_id, occurred, aware, reported, points, death, widespread, summary):
    """One candidate incident. Offsets are days before AS_OF; the clock fields are ISO."""
    return {
        "incident_id": incident_id,
        "occurred_ts": iso(AS_OF_DT.replace(hour=9) - timedelta(days=occurred)),
        "aware_ts": iso(AS_OF_DT.replace(hour=14) - timedelta(days=aware)),
        "reported_ts": None if reported is None else
        iso(AS_OF_DT.replace(hour=16) - timedelta(days=reported)),
        "points": points, "death": death, "widespread": widespread, "summary": summary,
    }


INCIDENT_CANDIDATES = [
    _inc("I-01", 41, 22, 12, ("c",), False, False,
         "scoring inversion in production for six hours; refusals issued on reversed ranks"),
    _inc("I-02", 6, 5, None, ("b",), False, False,
         "clearing-house scoring API returned garbage; a payment rail was suspended 9 hours"),
    _inc("I-03", 12, 3, None, ("a",), True, False,
         "an applicant died; the refusal is cited as a contributing factor"),
    _inc("I-04", 9, 4, None, ("c",), False, True,
         "one region systematically mis-scored across eleven deployers"),
    _inc("I-05", 7, 7, None, (), False, False,
         "batch job crashed; 400 applications re-queued, no decision issued to anyone"),
    _inc("I-06", 8, 6, 3, ("a", "b"), True, False,
         "load-ranking run on our stack took two substations off for four hours; one death"),
    _inc("I-07", 30, 28, 20, ("d",), False, False,
         "stale index feed; 300 applications scored on old data, one repossession claim"),
    _inc("I-08", 20, 19, None, ("a",), False, False,
         "refusal cascade; one applicant hospitalised with a documented causal link"),
]
print(f"{len(INCIDENT_CANDIDATES)} candidates")
for _c in INCIDENT_CANDIDATES:
    print(f"  {_c['incident_id']}  points={str(_c['points']):12s} death={_c['death']!s:5s} "
          f"widespread={_c['widespread']!s:5s} {_c['summary'][:46]}")

In [ ]:
def classify_incident(candidate: dict) -> dict:
    """Decide whether a candidate is a serious incident and which clock it runs on.

    `candidate["points"]` holds the Article 3(49) points the harm falls under, as letters.
    An empty tuple means no Article 3(49) harm, so it is not a serious incident.

    Decide in THIS order:
      1. no points at all -> not serious: paragraph None, deadline_days None
      2. `widespread` is true, OR "b" is among the points -> "73(3)", 2 days
      3. `death` is true -> "73(4)", 10 days
      4. otherwise -> "73(2)", 15 days

    Step 2 comes before step 3 on purpose. A death that is also a critical-infrastructure
    disruption gets the SHORTER deadline; see the note above, which says why and says that
    it is this lesson's reading rather than the Regulation's.

    Returns {"incident_id", "serious", "points", "paragraph", "deadline_days", "trigger"},
    where "points" is the sorted tuple of letters and "trigger" is a short string naming what
    decided the branch — the words a reviewer needs to disagree with you.

    Example:
        >>> classify_incident({"incident_id": "X", "points": ("c",), "death": False,
        ...                    "widespread": False})["deadline_days"]
        15
        >>> classify_incident({"incident_id": "Y", "points": ("a",), "death": True,
        ...                    "widespread": False})["paragraph"]
        '73(4)'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_classify() -> None:
    by_id = {c["incident_id"]: classify_incident(c) for c in INCIDENT_CANDIDATES}
    assert by_id["I-05"]["serious"] is False, (
        "I-05 issued no decision and caused none of the four Article 3(49) harms; it is an "
        "outage, not a serious incident"
    )
    assert by_id["I-05"]["paragraph"] is None and by_id["I-05"]["deadline_days"] is None, (
        "a candidate that is not a serious incident has no Article 73 paragraph and no clock"
    )
    assert by_id["I-01"]["deadline_days"] == 15, (
        f"I-01 is a point (c) fundamental-rights infringement and gets {DEADLINE_DAYS['73(2)']}"
        " days under 73(2). The two-day deadline belongs to a widespread infringement or to "
        "point (b), critical infrastructure — not to (c)"
    )
    assert by_id["I-02"]["paragraph"] == "73(3)" and by_id["I-02"]["deadline_days"] == 2, (
        "I-02 is point (b), a serious and irreversible disruption of critical infrastructure: "
        "two days under 73(3)"
    )
    assert by_id["I-03"]["paragraph"] == "73(4)" and by_id["I-03"]["deadline_days"] == 10, (
        "I-03 is a death with no critical-infrastructure element: ten days under 73(4)"
    )
    assert by_id["I-04"]["deadline_days"] == 2, (
        "I-04 is a point (c) harm, which alone would be fifteen days — but it is a WIDESPREAD "
        "infringement, and 73(3) names that separately from the Article 3(49) points"
    )
    assert by_id["I-06"]["paragraph"] == "73(3)", (
        "I-06 is both a death and a critical-infrastructure disruption. This lesson takes the "
        "shorter deadline; check the order of your branches"
    )
    assert by_id["I-08"]["deadline_days"] == 15, (
        "I-08 is serious harm to health, which is point (a) — but 73(4)'s ten-day clock is "
        "for the DEATH of a person, not for every point (a) harm"
    )
    assert by_id["I-06"]["points"] == ("a", "b"), "points comes back sorted"
    assert all(len(v["trigger"]) > 10 for v in by_id.values()), (
        "every classification names what decided it; a reviewer cannot argue with an empty "
        "string"
    )
    counts = {}
    for v in by_id.values():
        counts[v["deadline_days"]] = counts.get(v["deadline_days"], 0) + 1
    print(f"exercise 6 looks right — deadlines {counts} across {len(by_id)} candidates")

In [ ]:
_try("exercise 6", _check_classify)

## 9. Exercise 7 — the incident log, and the clock that starts at awareness

Every one of those deadlines runs from the moment the provider or the deployer **became
aware**, and in this fixture awareness lags occurrence by anything from two hours to nine
days. Starting the clock at occurrence feels conservative. It is not conservative; it is
wrong, and it is wrong in a way that will have you telling a regulator you were late when
you were not, and re-prioritising the wrong case today.

The log below is the artefact. It sorts by deadline, so the thing that is due first is read
first — not the thing that happened first.

<details><summary>💡 Hint 1 — what to think about</summary>

Every deadline runs from AWARENESS, and in this fixture awareness trails occurrence by days,
so the start of the clock changes verdicts. Then think about truncation: `int()` rounds
towards zero, which is the wrong direction for a deadline already behind you. Finally, the
log is read by whoever has to act first — so what is the sort key, and what breaks a tie?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Classify each candidate with your exercise 6 function; count and skip the ones that are not
serious. For the rest, the due moment is awareness plus the deadline in days. Status: if
reported, compare the report with the due moment ("not later than", so equal is in time); if
not, compare `as_of_dt` the same way. Both hour figures are seconds over 3600, floored
towards minus infinity. Sort on `(due_ts, incident_id)`, start the counts at zero for all
four statuses, and fail the verdict on any late or overdue entry.
</details>

In [ ]:
def incident_log(candidates: list, as_of_dt: datetime = AS_OF_DT) -> dict:
    """Build the Article 73 incident log, with a clock on every entry.

    Classify each candidate with `classify_incident`. Skip the ones that are not serious,
    counting them in "n_not_serious". For each serious one, an entry:

      "incident_id", "paragraph", "deadline_days", "points", "trigger", "summary"
      "occurred_ts", "aware_ts", "reported_ts"
      "due_ts"                 aware_ts + deadline_days days, as an ISO string
      "awareness_lag_hours"    whole hours from occurrence to awareness
      "hours_remaining"        whole hours from `as_of_dt` to due_ts, negative when past
      "status"                 "reported_in_time"  reported at or before due_ts
                               "reported_late"     reported after due_ts
                               "open_in_time"      not reported, as_of_dt at or before due_ts
                               "open_overdue"      not reported, as_of_dt after due_ts

    Whole hours are truncated towards minus infinity, so an entry two hours past its deadline
    reports -2 and never -1.

    Returns {"as_of", "entries", "counts", "n_not_serious", "verdict"}. "entries" is sorted by
    ("due_ts", "incident_id") — soonest deadline first. "counts" tallies the four statuses,
    with a 0 for any that does not occur. The verdict is "fail" when anything is
    "reported_late" or "open_overdue", else "pass".

    Example:
        >>> log = incident_log(INCIDENT_CANDIDATES)     # doctest: +SKIP
        >>> log["entries"][0]["incident_id"], log["n_not_serious"]
        ('I-07', 1)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_log() -> None:
    log = incident_log(INCIDENT_CANDIDATES)
    assert log["n_not_serious"] == 1, (
        f"one candidate is not a serious incident; you counted {log['n_not_serious']}"
    )
    assert len(log["entries"]) == 7, (
        f"{len(log['entries'])} entries; seven of the eight candidates are serious"
    )
    order = [e["incident_id"] for e in log["entries"]]
    assert order == ["I-07", "I-01", "I-06", "I-08", "I-02", "I-04", "I-03"], (
        f"entries came out {order}. Sort by due_ts — soonest first — and break the tie "
        "between I-06 and I-08, which are due at the same minute, by incident_id"
    )
    by_id = {e["incident_id"]: e for e in log["entries"]}
    assert by_id["I-01"]["status"] == "reported_in_time", (
        "I-01 was reported ten days after awareness against a fifteen-day deadline. If yours "
        "says late, you started the clock at the occurrence: the gap between the two here is "
        f"{by_id['I-01']['awareness_lag_hours'] / 24:.0f} days"
    )
    assert by_id["I-03"]["status"] == "open_in_time", (
        "I-03 is a death, aware three days ago, ten-day clock — still open and still in time. "
        "Starting at occurrence makes it look overdue"
    )
    assert by_id["I-02"]["status"] == "open_overdue" and by_id["I-04"]["status"] == "open_overdue", (
        "I-02 and I-04 both run on the two-day clock and both are past it"
    )
    assert by_id["I-06"]["status"] == "reported_late", (
        "I-06 was reported three days after awareness against the two-day clock this lesson "
        "gives it. On the ten-day clock it would look fine, which is what the branch order in "
        "exercise 6 decides"
    )
    assert by_id["I-08"]["hours_remaining"] == -94, (
        f"I-08's hours_remaining is {by_id['I-08']['hours_remaining']}, expected -94 — whole "
        "hours from as_of to the deadline, truncated downwards, negative because it is past"
    )
    assert by_id["I-03"]["awareness_lag_hours"] == 221, (
        f"I-03's awareness lag is {by_id['I-03']['awareness_lag_hours']}h, expected 221 — nine "
        "days and five hours between the death and the day anybody here heard about it"
    )
    assert log["counts"] == {"reported_in_time": 2, "reported_late": 1,
                             "open_in_time": 1, "open_overdue": 3}, (
        f"counts came out {log['counts']}; report all four statuses, with a 0 for any that "
        "does not occur"
    )
    assert log["verdict"] == "fail", "four entries are late or overdue, so the log fails"
    print(f"exercise 7 looks right — {len(log['entries'])} entries, {log['counts']}, verdict "
          f"{log['verdict']}")

In [ ]:
_try("exercise 7", _check_log)

In [ ]:
def _show_wrong_clocks() -> None:
    """Price the two classic clock mistakes on this fixture, rather than asserting them."""
    right = {e["incident_id"]: e["status"] for e in incident_log(INCIDENT_CANDIDATES)["entries"]}

    from_occurrence = [dict(c, aware_ts=c["occurred_ts"]) for c in INCIDENT_CANDIDATES]
    occ = {e["incident_id"]: e["status"]
           for e in incident_log(from_occurrence)["entries"]}
    flipped = sorted(k for k in right if right[k] != occ[k])
    print(f"clock started at OCCURRENCE instead of awareness: {len(flipped)} of {len(right)} "
          f"verdicts change {flipped}")
    for k in flipped:
        print(f"    {k}: {right[k]:17s} -> {occ[k]}")

    flat = [dict(c, points=("c",) if c["points"] else (), death=False, widespread=False)
            for c in INCIDENT_CANDIDATES]
    ffl = {e["incident_id"]: e["status"] for e in incident_log(flat)["entries"]}
    flipped2 = sorted(k for k in right if right[k] != ffl[k])
    print(f"\nevery serious incident given the 15-day default: {len(flipped2)} verdicts "
          f"change {flipped2}")
    for k in flipped2:
        print(f"    {k}: {right[k]:17s} -> {ffl[k]}")
    print("\nBoth mistakes are silent. Both produce a clean-looking log. One of them tells a")
    print("regulator you were late when you were not; the other tells you that you are fine.")


_try("wrong clocks", _show_wrong_clocks, needs=("exercise 6", "exercise 7"))

## 10. The artefact

A monitoring plan with computed control limits, and an incident log with clocks. This is the
object behind the conformity pack's `post_market_monitoring_plan` and
`serious_incident_procedure` — and unlike the paragraph it replaces, it can be re-run next
quarter against next quarter's feed and diffed.

In [ ]:
def monitoring_plan(k: float = DEFAULT_K) -> dict:
    """Assemble the Article 72(3) plan. Given to you — it is your own functions in a row."""
    limits = control_limits(BASELINE_ERRORS, BASELINE_N, WINDOW_SIZE, k)
    offset = estimate_clock_offset(PROVIDER_FEED, DEPLOYER_FEED)
    ingest = merge_deployer_feed(PROVIDER_FEED, DEPLOYER_FEED, offset["offset_seconds"])
    own = windowed_monitor(PROVIDER_FEED, limits)
    both = windowed_monitor(ingest["merged"], limits)
    return {
        "system_id": SYSTEM_ID, "as_of": iso(AS_OF_DT),
        "metric": "share of decisions later found wrong",
        "window_size": WINDOW_SIZE, "min_run": MIN_RUN,
        "baseline": {"windows": BASELINE_WINDOWS, "errors": BASELINE_ERRORS, "n": BASELINE_N},
        "limits": limits,
        "sources": {"provider_rows": ingest["n_provider"],
                    "deployer_rows": ingest["n_deployer"],
                    "deployer_only_rows": ingest["n_added"],
                    "clock_offset_seconds": offset["offset_seconds"],
                    "clock_spread_seconds": offset["spread_seconds"],
                    "rows_out_of_sequence": ingest["out_of_seq_rows"]},
        "alerts": {"provider_only": own["alerts"], "with_deployer": both["alerts"]},
        "cost": monitor_costs(own, DEGRADED_WINDOWS),
    }


def _show_artefact() -> None:
    plan = monitoring_plan()
    log = incident_log(INCIDENT_CANDIDATES)
    print(f"post-market monitoring plan · {plan['system_id']} · {plan['as_of']}")
    print(f"  metric        {plan['metric']}")
    print(f"  window        {plan['window_size']} decisions, confirm over {plan['min_run']}")
    print(f"  baseline      windows 0..{plan['baseline']['windows'] - 1}, "
          f"{plan['baseline']['errors']}/{plan['baseline']['n']}")
    print(f"  centre / ucl  {plan['limits']['centre']:.4f} / {plan['limits']['ucl']:.4f} "
          f"at k={plan['limits']['k']}")
    for key, value in plan["sources"].items():
        print(f"  {key:22s} {value}")
    print(f"  alerts        own {plan['alerts']['provider_only']}, with the deployer "
          f"{plan['alerts']['with_deployer']}")
    print(f"  cost at k={plan['limits']['k']}  {plan['cost']['total_cost']} "
          f"({plan['cost']['false_alerts']} false, "
          f"{len(plan['cost']['missed_windows'])} missed)")
    print(f"\nserious incident log · {log['as_of']} · verdict {log['verdict'].upper()}")
    print(f"  {'id':6s} {'para':7s} {'days':>4s} {'due':21s} {'left h':>7s}  status")
    for entry in log["entries"]:
        print(f"  {entry['incident_id']:6s} {entry['paragraph']:7s} "
              f"{entry['deadline_days']:>4d} {entry['due_ts']:21s} "
              f"{entry['hours_remaining']:>7d}  {entry['status']}")
    print(f"  {log['n_not_serious']} candidate(s) classified as not a serious incident")


_try("the artefact", _show_artefact, needs=tuple(_EXERCISES))

### The two halves are one artefact

The monitor and the incident log are usually built by different people in different
quarters. They are the same duty. The cell below asks the one question that joins them:
when did the monitor know, against when the incident record says anybody was aware?

In [ ]:
def _show_awareness_gap() -> None:
    plan = monitoring_plan()
    limits = plan["limits"]
    offset = estimate_clock_offset(PROVIDER_FEED, DEPLOYER_FEED)
    ingest = merge_deployer_feed(PROVIDER_FEED, DEPLOYER_FEED, offset["offset_seconds"])
    both = windowed_monitor(ingest["merged"], limits)
    own = set(windowed_monitor(PROVIDER_FEED, limits)["alerts"])
    new = [a for a in both["alerts"] if a not in own]
    if not new:
        print("no deployer-only alert to compare against")
        return
    confirmed = both["windows"][new[0] + MIN_RUN - 1]["last_ts"]
    log = incident_log(INCIDENT_CANDIDATES)
    widespread = [e for e in log["entries"] if e["paragraph"] == "73(3)"
                  and e["incident_id"] == "I-04"][0]
    gap = (parse_ts(widespread["aware_ts"]) - parse_ts(confirmed)).total_seconds() / 3600
    print(f"the merged monitor's alert was confirmed at {confirmed}")
    print(f"I-04's recorded awareness is       {widespread['aware_ts']}")
    print(f"gap: {gap:.0f} hours, or {gap / 24:.1f} days.")
    print("\nArticle 73's clock starts when you become aware. A monitor that alerted and was")
    print("not read is a poor argument that nobody was aware, and the gap above is the size")
    print("of that argument, in hours, on the face of the artefact.")


_try("awareness gap", _show_awareness_gap, needs=tuple(_EXERCISES))

## 11. Common mistakes

- **Refitting the control limits after the excursion.** The centre line moves, the limits
  widen, and the breach sits inside them. Nothing was falsified. Declare the baseline
  window, record it in the plan, and change it only in a change record with a date on it.
- **Dividing by the baseline size instead of the window size.** The limits then describe how
  precisely you know the baseline mean, which is not the question. The question is how far a
  window of `WINDOW_SIZE` decisions wanders when nothing is wrong.
- **Scoring the incomplete tail.** The last seventeen decisions in this feed have a rate
  several times the upper limit and mean nothing at all.
- **Ordering the log by timestamp.** Sequence numbers are the authority inside one log. The
  batch the chart cell in section 4 counts is not an exotic case: late-arriving data is the
  normal condition of a monitoring feed.
- **Paging on one window.** And its mirror image, confirming over so many windows that the
  alert arrives after the quarter ends. Both are settings; both should be in the plan with a
  number beside them.
- **Tuning `k` by habit.** `k = 3` is a convention from a different problem with different
  costs. The sweep in section 5 finds a cheaper setting on this feed, and a different one
  again when the cost ratio changes.
- **Counting only one kind of error.** A monitor tuned for zero false alerts has a cost; it
  is simply paid by somebody else, later, and it does not appear on the monitoring team's
  ledger. Put both on the same ledger.
- **Taking the mean of a clock offset.** Four re-exported rows out of a hundred moved the
  mean by the better part of an hour here. The median did not move.
- **Appending the whole deployer feed.** The rows you already hold are not new evidence.
  They dilute exactly the rate you were trying to measure.
- **Giving a fundamental-rights infringement the two-day deadline.** It is point (c) and it
  gets fifteen days; the two-day clock is for a widespread infringement or for point (b).
  The same confusion, run the other way, gives a critical-infrastructure disruption a
  fortnight it does not have.
- **Starting the reporting clock at occurrence.** Or at the day the root cause was found.
  Article 73 says awareness, and the initial report may be incomplete precisely so that the
  clock can be met before the investigation is finished.

In [ ]:
def _show_without_the_plan() -> None:
    """What the same evidence looks like written the way most packs write it."""
    plan = monitoring_plan()
    log = incident_log(INCIDENT_CANDIDATES)
    print("The same quarter, as a monitoring section usually reads:\n")
    print('  "Performance is monitored continuously. No significant degradation was')
    print('   observed. Serious incidents are reported in accordance with Article 73."\n')
    print("Everything in those two sentences is true of this feed. What they leave out:")
    print(f"  · the limits            centre {plan['limits']['centre']:.4f}, ucl "
          f"{plan['limits']['ucl']:.4f}, k={plan['limits']['k']}, fitted to "
          f"{plan['baseline']['n']} declared decisions")
    print(f"  · what 'continuously' means   windows of {plan['window_size']}, confirmed over "
          f"{plan['min_run']}")
    print(f"  · the alerts            {plan['alerts']['with_deployer']}")
    print(f"  · what the setting cost {plan['cost']['total_cost']}")
    print(f"  · the other clock       {plan['sources']['clock_offset_seconds']:.0f}s off, "
          f"{plan['sources']['rows_out_of_sequence']} rows out of sequence")
    print(f"  · the incident log      {log['counts']}")
    print("\nSame quarter, same data, same verdict. One of them can be re-run.")


_try("without the plan", _show_without_the_plan, needs=tuple(_EXERCISES))

## 12. Self-check

1. A high-risk system infringes an obligation under Union law intended to protect
   fundamental rights. You become aware on 1 June and establish the causal link on 4 June.
   Article 73 requires the report:
   - (a) within two days of 1 June, because fundamental rights are the most urgent category
   - (b) within two days of 4 June, when the link was established
   - (c) immediately once the link was established, and in any event within fifteen days of
         1 June: the two-day deadline is for a widespread infringement or for point (49)(b)
   - (d) within ten days of 1 June

2. An incident occurred on 1 March. Your deployer told you on 20 March. You established the
   causal link on 22 March. The fifteen-day long stop in Article 73(2) runs from:
   - (a) 1 March, the occurrence
   - (b) 20 March, when you became aware — the causal link governs the "immediately" duty,
         not the long stop
   - (c) 22 March, when the link was established
   - (d) whichever of the three is earliest

3. Your control chart has been quiet for a quarter, so you recompute the limits over the
   last twelve months, which include a degradation you already know about. What happens?
   - (a) nothing much: the centre line moves but the limits are unchanged
   - (b) the limits narrow and you get more alerts
   - (c) the chart becomes more accurate, because it now uses more data
   - (d) the limits widen until the degradation sits inside them, and the alert disappears —
         which is what the refit demo at the end of section 4 measures

4. A monitor pages three times in a quarter for excursions that turn out to be nothing, and
   the team mutes it. On this lesson's cost model, the bill for those three pages is:
   - (a) three investigations, and nothing else
   - (b) zero, because nothing was actually wrong
   - (c) three investigations plus every degraded window that happened afterwards, because a
         muted monitor detects nothing
   - (d) unknowable without the false positive rate

5. What did Regulation (EU) 2026/1744 change in Article 72?
   - (a) it replaced the implementing act that was to establish a post-market monitoring
         plan template by 2 February 2026 with Commission guidance, including a template,
         by 2 September 2027
   - (b) it deleted Article 72(2)'s reference to data provided by deployers
   - (c) it shortened the Article 73 deadlines to two days across the board
   - (d) nothing; Article 72 is as adopted in 2024

Mark them in the next cell. The key is not written in this file — only a salted hash of it —
so you find out which are wrong without reading the answers off the page.

In [ ]:
# Salted hashes of the answers, not the answers. Nothing here tells you which letter is right.
_SELF_CHECK_KEY = {
    1: "27c10c010067e766",
    2: "634cd61330a9d618",
    3: "570691b600383589",
    4: "9182297f4ef6b404",
    5: "ba0238ee6a71462a",
}

_SELF_CHECK_HINT = {
    1: "read the ARTICLE_3_49 table in section 1 and ask which point this harm is.",
    2: "read the sentence under 'Three deadlines, one starting gun'.",
    3: "run the refit demo at the end of section 4 and compare the two alert lists.",
    4: "look at what `switched_off_at` does to `missed_windows` in the k sweep.",
    5: "look at the OMNIBUS_NOTES table and at what art72_3 says was replaced with what.",
}


def check_self_check(answers: dict) -> None:
    """Mark your self-check answers. Pass a dict of question number -> letter.

    Example:
        >>> check_self_check({1: "a"})          # doctest: +SKIP
          q1  not 'a' — read the ARTICLE_3_49 table in section 1 ...
          q2  no answer given
        ...
    """
    right = 0
    for question in sorted(_SELF_CHECK_KEY):
        given = str(answers.get(question, "")).strip().lower()
        digest = hashlib.sha256(f"P01-L07:q{question}:{given}".encode()).hexdigest()[:16]
        if digest == _SELF_CHECK_KEY[question]:
            right += 1
            print(f"  q{question}  correct")
        elif not given:
            print(f"  q{question}  no answer given")
        else:
            print(f"  q{question}  not {given!r} — {_SELF_CHECK_HINT[question]}")
    print(f"\n{len(_SELF_CHECK_KEY)} questions, {right} right")


# Put your own letters in, then run this cell:
# check_self_check({1: "a", 2: "a", 3: "a", 4: "a", 5: "a"})

## What you built, and where it goes next

A monitoring plan whose every number was computed — a declared baseline, limits fitted to
it, a window size, a confirmation rule, and a `k` with a bill attached — plus an ingest path
that reconciles somebody else's clock and says how far it trusted it, and an incident log
whose deadlines start where Article 73 says they start.

The pieces travel. The control limits are the thing module 8 checks before it lets a
declaration of conformity be emitted, and the thing the capstone's inspector asks to see the
fitting window for. The ingest path is where module 5's oversight records arrive from when
the reviewers are the deployer's staff and not yours. And the habit — every setting carrying
the cost of being wrong in both directions — is what separates a monitoring plan from a
paragraph that says monitoring happens.

**Again, and finally: this is engineering, not legal advice.**

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_control_limits),
                              ("exercise 2", _check_monitor),
                              ("exercise 3", _check_costs),
                              ("exercise 4", _check_offset),
                              ("exercise 5", _check_merge),
                              ("exercise 6", _check_classify),
                              ("exercise 7", _check_log)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))